# Amortized Inference for a NLME Model

In [ ]:
# load necessary packages
import numpy as np
import pandas as pd
import itertools
from pypesto import optimize

# for plots
from IPython.display import display
from heatmap import corrplot

from inference.inference_functions import run_population_optimization, create_boundaries_from_prior
from inference.nlme_objective import ObjectiveFunctionNLME
from inference.helper_functions import transform_pesto_results, load_multi_experiment_data
from inference.ploting_routines import plot_real_vs_synthetic, plot_real_and_estimated, \
    plot_parameter_estimates, visualize_pesto_result

In [ ]:
# specify which model to use
model_name = ['fröhlich-small', 'fröhlich-large', 'fröhlich-sde'][2]

## Load model

In [ ]:
if model_name == 'fröhlich-small':
    from models.froehlich_model_small import FroehlichModelSmall
    model = FroehlichModelSmall(load_best=True)

    use_presimulation = False
elif model_name == 'fröhlich-large':
    from models.froehlich_model_large import FroehlichModelLarge
    model = FroehlichModelLarge(load_best=True)

    use_presimulation = True
    presimulation_path = 'data/presimulations_froehlich_large'

elif model_name == 'fröhlich-sde':
    from models.froehlich_model_sde import FroehlichModelSDE
    model = FroehlichModelSDE(load_best=True)

    use_presimulation = True
    presimulation_path = 'data/presimulations_froehlich_sde'
else:
    raise NotImplementedError('model not implemented')

path_store_network = 'networks/' + model.network_name

# assemble simulator and prior
simulator =  model.build_simulator()
trainer = model.build_trainer(path_store_network)

In [ ]:
model.print_and_plot_example()

## Estimating Population Parameters

Now we want to use the amortizer to generate samples such that we can minimize the negative log-likelihood of the data given the population parameters of the mixed effect model:
$$
    \beta^*,\Psi^* \approx 
    \underset{\beta,\Psi}{\arg\min} -\sum_i \log\left( \frac1M \sum_j^M \frac{p(\phi_j \mid \beta,\Psi)}{p(\phi_j)} \right).
$$

Remark: the objective value is not the likelihood value since the sum over $\log p(y_i)$ is missing.
One part of the objective function increases logarithmically with the sample size $M$.
On the other hand, the approximation gets better (therby decreasing the value?!)

$\beta$ is called ```theta_population``` in the code.


$\log \phi$ cell specific parameters, sampled from $\mathcal{N}(\beta,\Psi)$
$$
    p( \phi \mid \beta,\Psi) = (2\pi)^{-k/2}\vert \Psi\vert^{-1/2} \prod_{l=1}^k \phi_l^{-1} \exp \left(-\frac12 (\log \phi-\beta)^T \Psi^{-1}  (\log \phi-\beta) \right)
$$

Assumptions to start with: $\Psi$ is a diagonal matrix, need better parameterization for other types

$$
    \beta^*,\Psi^* \approx
    \underset{\beta,\Psi}{\arg\min} -\sum_i \left(\log \frac1M \sum_j^M \frac{p( \phi_j \mid \beta,\Psi)}{p( \phi_j)} \right) \\
     =  \underset{\beta,\Psi}{\arg\min} -\sum_i \left(\log\left(\vert \Psi\vert^{-1/2} \right) -\log M -
    \log\left(\vert \Sigma\vert^{-1/2}\right) +\log \sum_j^M \exp \left(-\frac12 (\log\phi_j-\beta)^T \Psi^{-1}  (\log\phi_j-\beta) + \frac12 (\log\phi_j-\mu)^T \Sigma^{-1}  (\log\phi_j-\mu) \right)\right)
$$

if the prior is $p( \phi) = (2\pi)^{-k/2}\vert \Sigma\vert^{-1/2} \prod_{l=1}^k \phi_l^{-1}\exp \left(-\frac12 (\log \phi-\mu)^T \Sigma^{-1}  (\log\phi-\mu) \right)$.


For purpose of optimization we also parametrize $\Psi$ by a log-transformation since diagonal entries must be positive.

# Define Objective Function

In [ ]:
cov_type = ['diag', 'cholesky'][1]
obj_fun_amortized = ObjectiveFunctionNLME(model_name=model.name,
                                          param_samples=np.empty((1,1,1)),
                                          prior_mean=model.prior_mean,
                                          prior_std=model.prior_std,
                                          covariance_format=cov_type,
                                          # np.exp(-3.41) * 1.5  # 1.5 times the median of the posterior standard deviations
                                          huber_loss_delta=None)

obj_fun_amortized_2 = ObjectiveFunctionNLME(model_name=model.name,
                                          param_samples=np.empty((1,1,1)),
                                          prior_mean=model.prior_mean,
                                          prior_std=model.prior_std,
                                          covariance_format=cov_type,
                                          # np.exp(-3.41) * 1.5  # 1.5 times the median of the posterior standard deviations
                                          huber_loss_delta=None)

## Apply on Real Data (Multi-Experiment for Fröhlich Model)

In [ ]:
param_names = model.param_names.copy()

# create boundaries
lower_bound, upper_bound = create_boundaries_from_prior(
        prior_mean=model.prior_mean,
        prior_std=model.prior_std,
        boundary_width_from_prior=2.58,
        covariance_format='diag' # cholesky is created later
)

results_to_compare = None

if model_name == 'fröhlich-small':
    param_names.pop(1)
    param_names.insert(1, '$\gamma_{eGFP}$')
    param_names.insert(2, '$\gamma_{d2eGFP}$')

    lower_bound = np.concatenate((lower_bound[:2], [lower_bound[1]],
                                  lower_bound[2:8],  [lower_bound[7]],  lower_bound[8:]))
    upper_bound = np.concatenate((upper_bound[:2], [upper_bound[1]],
                                  upper_bound[2:8],  [upper_bound[7]],  upper_bound[8:]))

elif model_name == 'fröhlich-large':
    param_names.pop(7)
    param_names.insert(7, '$\gamma_{eGFP}$')
    param_names.insert(8, '$\gamma_{d2eGFP}$')

    lower_bound = np.concatenate((lower_bound[:8], [lower_bound[7]],
                                  lower_bound[8:19], [lower_bound[18]], lower_bound[19:]))
    upper_bound = np.concatenate((upper_bound[:8], [upper_bound[7]],
                                  upper_bound[8:19], [upper_bound[18]], upper_bound[19:]))
elif model_name == 'fröhlich-sde':
    param_names.pop(1)
    param_names.insert(1, '$\gamma_{eGFP}$')
    param_names.insert(2, '$\gamma_{d2eGFP}$')

    lower_bound = np.concatenate((lower_bound[:2], [lower_bound[1]],
                                  lower_bound[2:10],  [lower_bound[9]], lower_bound[10:]))
    upper_bound = np.concatenate((upper_bound[:2], [upper_bound[1]],
                                  upper_bound[2:10],  [upper_bound[9]], upper_bound[10:]))

else:
    raise NotImplementedError("Model Data not available")

pop_param_names = ['pop-' + name for name in param_names]
var_param_names = ['var-' + name for name in param_names]

full_param_names = pop_param_names + var_param_names

# sigma variance is always fixed
index_sigma = [ix for ix, x in enumerate(full_param_names) if 'var-$\\sigma$' == x]
fixed_indices = np.array(index_sigma)  # variance of sigma fixed
fixed_vals = np.array([upper_bound[index_sigma[0]]])
n_params_multi = len(param_names)

In [ ]:
if cov_type == 'cholesky' and len(full_param_names) == model.n_params*2+2:
    # add  correlation names to full parameter names
    # make all possible combinations of parameter names
    combinations = list(itertools.combinations(param_names, 2))
    # create upper triangular matrix of parameter names
    psi_inverse_upper = np.chararray((n_params_multi, n_params_multi), itemsize=100, unicode=True)
    psi_inverse_upper[np.diag_indices(n_params_multi)] = "1"
    psi_inverse_upper[np.triu_indices(n_params_multi, k=1)] = [f"corr_{x}_{y}" for x, y in combinations]
    # extract lower triangular matrix of parameter names, so that they are ordered as in the covariance matrix
    corr_names = list(psi_inverse_upper.T[np.tril_indices(n_params_multi, k=-1)])
    # add correlation names to full parameter names
    full_param_names = full_param_names + corr_names
    full_param_names.pop([ix for ix, name in enumerate(full_param_names) if name == 'corr_$\\gamma_{eGFP}$_$\\gamma_{d2eGFP}$'][0])
    print(full_param_names)

# Analyze Correlations

In [ ]:
obs_data_multi = load_multi_experiment_data(data_points=100)
print('eGFP:', obs_data_multi[0].shape[0])
print('d2eGFP:', obs_data_multi[1].shape[0])

In [ ]:
param_samples = model.draw_posterior_samples(data=obs_data_multi[0], n_samples=1000)
param_median = np.median(param_samples, axis=1)
# correlations without sigma
names = model.param_names[:-1]
names[[ix for ix, name in enumerate(model.param_names[:-1]) if'gamma' in name][0]] = '$\gamma_{eGFP}$'
median_df = pd.DataFrame(param_median[:, :-1], columns=names)
corr_df = median_df.corr()
print('Correlation Matrix from Median - eGFP')
corrplot(corr_df, size_scale=300)

In [ ]:
param_samples = model.draw_posterior_samples(data=obs_data_multi[1], n_samples=1000)
param_median = np.median(param_samples, axis=1)
# correlations without sigma
names = model.param_names[:-1]
names[[ix for ix, name in enumerate(model.param_names[:-1]) if'gamma' in name][0]] = '$\gamma_{d2eGFP}$'
median_df_d2 = pd.DataFrame(param_median[:, :-1], columns=names)
corr_df_d2 = median_df_d2.corr()
print('Correlation Matrix from Median - d2eGFP')
corrplot(corr_df_d2, size_scale=300)

In [ ]:
if cov_type == 'cholesky':
    # Find pairs with correlation above the threshold
    threshold = 0
    abs_corr_matrix = corr_df.abs()
    high_corr_pairs = abs_corr_matrix[abs_corr_matrix > threshold].unstack()
    high_corr_pairs = high_corr_pairs[high_corr_pairs < 1].sort_values(ascending=False)

    abs_corr_matrix_d2 = corr_df_d2.abs()
    high_corr_pairs_d2 = abs_corr_matrix_d2[abs_corr_matrix_d2 > threshold].unstack()
    high_corr_pairs_d2 = high_corr_pairs_d2[high_corr_pairs_d2 < 1].sort_values(ascending=False)

    high_corr_pairs_joint = pd.concat([high_corr_pairs, high_corr_pairs_d2])

    # find indices corresponding to pairs with high correlations
    high_corr_pairs_index = []
    for x, y in high_corr_pairs_joint.index:
        name = f'corr_{x}_{y}'
        high_corr_pairs_index += [p_i for p_i, p_name in enumerate(full_param_names) if name == p_name]
    high_corr_pairs_index = np.unique(high_corr_pairs_index)
    high_corr_pairs_names = [full_param_names[i] for i in high_corr_pairs_index]
    print(high_corr_pairs_names, len(high_corr_pairs_index))

In [ ]:
only_allow_high_corr = True
if cov_type == 'cholesky':
    if only_allow_high_corr:
        # fix all correlations to 0 but high correlated ones
        index_all_corr = [ix for ix, x in enumerate(full_param_names) if 'corr' in x]
        # remove high_corr_pairs_index from index_all_corr
        index_all_corr = [ix for ix in index_all_corr if ix not in high_corr_pairs_index]
        fixed_indices = np.append(fixed_indices, index_all_corr)
        fixed_vals = np.append(fixed_vals, np.zeros(len(index_all_corr)))
    else:
        # fix all correlations with sigma to 0
        index_sigma_corr = [ix for ix, x in enumerate(full_param_names) if 'sigma' in x and 'corr' in x]
        fixed_indices = np.append(fixed_indices, index_sigma_corr)
        fixed_vals = np.append(fixed_vals, np.zeros(len(index_sigma_corr)))

In [ ]:
if obj_fun_amortized.covariance_format == 'cholesky':
    # add lower triangular part of covariance matrix
    lower_bound = np.concatenate((lower_bound, -2.58 * np.ones(len(full_param_names)-lower_bound.size)))
    upper_bound = np.concatenate((upper_bound, 2.58 * np.ones(len(full_param_names)-upper_bound.size)))

In [ ]:
index_offset = [ix for ix, x in enumerate(full_param_names) if 'offset' in x and 'pop' not in x]
index_scale = [ix for ix, x in enumerate(full_param_names) if 'scale' in x and 'pop' not in x]

fixed_indices = np.append(fixed_indices, index_offset+index_scale)  # fix offset and scale
fixed_vals = np.append(fixed_vals, upper_bound[index_offset+index_scale])

In [ ]:
# make sure that fixed indices are unique
fixed_indices, unique_indices = np.unique(fixed_indices, return_index=True)
fixed_vals = fixed_vals[unique_indices]

In [ ]:
#non_fix = [i for i, _ in enumerate(full_param_names) if i not in fixed_indices]
#np.array(full_param_names)[non_fix]

## Estimate Multiple Experiments Jointly

In [ ]:
%%time
result_optimization = run_population_optimization(
    bf_amortizer=model.amortizer,
    data=obs_data_multi,
    param_names=full_param_names,
    objective_function=[obj_fun_amortized, obj_fun_amortized_2],
    sample_posterior=model.draw_posterior_samples,
    n_multi_starts=20,
    noise_on_start_param=1,
    n_samples_opt=50,
    lb=lower_bound,
    ub=upper_bound,
    x_fixed_indices=fixed_indices,
    x_fixed_vals=fixed_vals,
    file_name=f'output/multi-experiment/{model_name}_cholesky_test_full.hd5', # 'Auto',
    verbose=True,
    pesto_multi_processes=10,
    multi_experiment=True)

results = result_optimization.optimize_result.as_dataframe()['x']

In [ ]:
#from pypesto.store import OptimizationResultHDF5Writer
#writer = OptimizationResultHDF5Writer(f'output/multi-experiment/{model_name}-test.hd5')
#writer.write(result_optimization)

In [ ]:
#save_name = 'output/multi-experiment/' + model_name + '_' + cov_type + '_' + str(0) + '_scipy'
#from pypesto.store import OptimizationResultHDF5Reader
#result_optimization = OptimizationResultHDF5Reader(save_name).read()
#results = result_optimization.optimize_result.as_dataframe()['x']
result_optimization.optimize_result.as_dataframe()

In [ ]:
visualize_pesto_result(result_optimization)

In [ ]:
param_idx_egfp = [i_n for i_n, name in enumerate(full_param_names) if 'd2eGFP' not in name]
param_idx_d2egfp = [i_n for i_n, name in enumerate(full_param_names) if
                                'd2eGFP' in name or 'eGFP' not in name]

results_egfp = results[0][param_idx_egfp]
results_d2egfp = results[0][param_idx_d2egfp]

In [ ]:
simulator = model.build_simulator(with_noise=False)

In [ ]:
estimated_beta = results_egfp[:n_params_multi-1]
estimated_psi = obj_fun_amortized.get_covariance(results_egfp[n_params_multi-1:])

print('eGFP')
plot_real_vs_synthetic(estimated_mean=estimated_beta,
                       estimated_cov=estimated_psi,
                       data=obs_data_multi[0],
                       model_name=model.name,
                       n_trajectories=len(obs_data_multi[0])*10,
                       simulator=simulator,
                       estimation_function=np.mean,
                       #save_fig=model_name+'_eGFP_dif_bayesflow_noisy',
                       ylim=(-1.,1.),
                       seed=0)
plot_real_and_estimated(estimated_mean=estimated_beta,
                        estimated_cov=estimated_psi,
                        data=obs_data_multi[0],
                        model_name=model.name,
                        n_trajectories=len(obs_data_multi[0]),
                        simulator=simulator,
                        #save_fig=model_name+'_eGFP_estimate_bayesflow_noisy',
                        seed=0)

In [ ]:
estimated_beta_d2 = results_d2egfp[:n_params_multi-1]
estimated_psi_d2 = obj_fun_amortized_2.get_covariance(results_d2egfp[n_params_multi-1:])

print('d2eGFP')
plot_real_vs_synthetic(estimated_mean=estimated_beta_d2,
                       estimated_cov=estimated_psi_d2,
                       data=obs_data_multi[1],
                       model_name=model.name,
                       n_trajectories=len(obs_data_multi[1])*10,
                       simulator=simulator,
                       #save_fig=model_name+'_eGFPd2_dif_byesflow_noisy',
                       ylim=(-1.,1.),
                       seed=0)
plot_real_and_estimated(estimated_mean=estimated_beta_d2,
                        estimated_cov=estimated_psi_d2,
                        data=obs_data_multi[1],
                        model_name=model.name,
                        n_trajectories=len(obs_data_multi[1]),
                        simulator=simulator,
                        #save_fig=model_name+'_eGFPd2_estimate_bayesflow_noisy',
                        seed=0)

In [ ]:
corr_df = pd.DataFrame(estimated_psi[:-1, :-1], columns=model.param_names[:-1], index=model.param_names[:-1])
print('Covariance Matrix for Random Effects')
corrplot(corr_df, size_scale=300)

In [ ]:
corr_df_d2 = pd.DataFrame(estimated_psi_d2[:-1, :-1], columns=model.param_names[:-1], index=model.param_names[:-1])
print('Covariance Matrix for Random Effects')
corrplot(corr_df_d2, size_scale=300)